# Reproducing Fig. 6 of arXiv:2606.20269 with `jexplore`

Bayesian inference of a **single resolved LISA galactic binary (GB)**, reproducing the
frequency-domain posterior of **Fig. 6 of [arXiv:2606.20269](https://arxiv.org/abs/2606.20269)**
with the affine-invariant ensemble sampler [`jexplore`](https://pypi.org/project/jexplore/).

The datastream (clean signal + instrumental noise) and the noise PSD come **entirely
from the helpers in [`canna/lisa.py`](canna/lisa.py)** — `clean_signal`, `sample_noise`
and `noise_psd` — the same code that feeds the flow-matching model in
[scripts/train.py](scripts/train.py).

## Paper setup

We match the paper's observation and injection:

* **band** `(1e-4, 3e-3)` Hz → cadence `dt = 166.7 s`, `N = 131072 = 2¹⁷` samples,
  `T_obs ≈ 253 d`;
* **injection** (Fig. 6 fiducials): `f₀ = 1.38599 mHz`, `ḟ = 9.39×10⁻¹⁵ Hz/s`,
  `A = 5.35×10⁻²⁴`, `φ₀ = 1.74`, ecliptic `λ = 4.50`, `β = 0.98`, polarization
  `ψ = 2.70`, inclination `ι = 0.58` rad;
* **optimal SNR ≈ 20.7** (A+E+T combined).

We sample the four parameters `θ = [f₀, ḟ, A, φ₀]` (in the code: `[log f₀, log ḟ, log A,
φ₀]`), holding the sky position, polarization and inclination fixed at the injected
values. The corner plot displays them in the paper's coordinates — `f₀` offset, `ḟ`,
and the amplitude quadratures `g_t = A cos φ₀`, `g_s = A sin φ₀`.

## Likelihood & SNR normalisation

`lisa.sample_noise` colours the frequency-domain noise to the **physical one-sided PSD**
`S(f) = lisa.noise_psd(freqs)[channel]`: each rfft bin has `E[|n_f|²] = (N/2·dt)·S(f)`,
so the matching Gaussian log-likelihood carries a `(2·dt/N)` factor:

`log L(θ) = −(2·dt/N) · Σ_f Σ_ch |d_f − h_f(θ)|² / S(f)`

At the truth this gives `log L ≈ −(#bins × #channels)` (χ², 2 dof per complex bin),
which we check below. The matched-filter optimal SNR is `√⟨h|h⟩`; since `canna.lisa`'s
`optimal_snr` uses the `(2·dt/N)` inner product it returns `√⟨h|h⟩ / √2`, so we report
`snr_phys = √2 × lisa.optimal_snr`, which recovers the paper's `≈ 20.7`.

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.abspath(""))   # so `import canna.lisa` works from the notebook
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")  # allocate on demand (shared GPU)

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import jax.random as jr
import numpy as np
import corner
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

from canna import lisa

from jexplore.sampler import JaxSampler, Steps
from jexplore.sampling import EpochMH, SamplingMH
from jexplore.steps import Stretch
from jexplore.backends import DefaultBackend

print("JAX backend:", jax.default_backend(), "| devices:", jax.devices())

## 1. Observation, true parameters and datastream

Everything below comes from `canna/lisa.py`. The clean A/E/T signal is
`lisa.clean_signal` and the instrumental noise realisation is `lisa.sample_noise`;
both return the rFFT on the same cropped frequency grid, so `data = signal + noise`.

In [ ]:
# ---- observation grid (arXiv:2606.20269 setup) ----
# The paper runs in the (1e-4, 3e-3) Hz band: cadence dt = 1/(2·3 mHz) = 166.7 s,
# N = 131072 = 2**17 samples, T_obs ≈ 253 d. The new canna.lisa hard-codes a 12 mHz
# band (dt ≈ 41.7 s) and fixes the observation grid as module globals at import;
# clean_signal / sample_noise / optimal_snr all read those globals (the old t_obs/dt
# arguments are gone). We override the grid ONCE here — before the first waveform is
# generated — so the functions run on the paper grid rather than the default one.
# (The grid is not just cosmetic: canna.optimal_snr scales with dt, so the 12 mHz grid
#  would give a ~4× different SNR for the same source.)
lisa.SAMPLING_STEP = 1.0 / (2.0 * 3e-3)               # 166.667 s  (paper band edge 3 mHz)
lisa.N_SAMPLES     = 1 << 17                          # 131072 samples (paper)
lisa.T_OBS         = lisa.N_SAMPLES * lisa.SAMPLING_STEP   # 21_845_333 s ≈ 253 d

# canna's clean_signal / optimal_snr are jitted and cache by argument shape, NOT by the
# module grid globals above. If this kernel already traced them on another grid (e.g. an
# earlier run), those stale traces would build arrays on the wrong grid and later crash
# the likelihood with a shape mismatch. Clearing the caches forces a re-trace on the grid
# we just set, so this cell is safe to re-run without restarting the kernel.
jax.clear_caches()

DT    = lisa.SAMPLING_STEP
T_OBS = lisa.T_OBS
SEED  = 0
freq  = jnp.fft.rfftfreq(lisa.N_SAMPLES, DT)

# Physical / matched-filter SNR = sqrt(<h|h>). canna.optimal_snr uses a (2·dt/N) inner
# product, which is 1/√2 of the <h|h> that the likelihood below (LOGLIK_NORM = 2·dt/N)
# and the paper both use, so the matched-filter SNR is √2 × lisa.optimal_snr.
snr_phys = lambda p: float(jnp.sqrt(2.0) * lisa.optimal_snr(p))

print(f"grid: dt={DT:.1f}s  N={lisa.N_SAMPLES}  T_obs={T_OBS/86400:.1f} d,  {freq.shape[0]} freq bins")

In [ ]:
# ---- INJECTION under test: the galactic binary of arXiv:2606.20269, Fig. 6 ----
# All eight parameters are the paper's fiducial values:
#     f0   = 1.38599 mHz   (log10 f0 = -2.85824130)
#     fdot = 9.39e-15 Hz/s
#     A    = 5.35e-24      (log10 A  = -23.2713149)
#     phi0 = 1.74 rad
#     ecliptic longitude λ = 4.50 rad,  ecliptic latitude β = 0.98 rad
#     polarization ψ = 2.70 rad,        inclination ι = 0.58 rad
# (canna's params order is [f0, fdot, A, ra, dec, psi, iota, phi0], where ra/dec are the
#  ecliptic longitude/latitude λ/β.)  Earlier versions of this notebook kept placeholder
# sky/orientation angles here; with the wrong inclination + sky position the injection
# SNR came out ~5× too low and the corner could not match the paper.

# [f0, fdot, A, ra(λ), dec(β), psi, iota, phi0]
params_new = jnp.array([10 ** -2.85824130,  9.38982036e-15,  10 ** -23.2713149,
                        4.50,  0.98,  2.70,  0.58,  1.74379807])

# make params_new the injected truth for the sampler below
F0_TRUE, FDOT_TRUE, A_TRUE = float(params_new[0]), float(params_new[1]), float(params_new[2])
RA_TRUE, DEC_TRUE          = float(params_new[3]), float(params_new[4])
PSI_TRUE                   = float(params_new[5])   # fixed (not inferred)
IOTA_TRUE, PHI0_TRUE       = float(params_new[6]), float(params_new[7])

# datastream = clean signal + physical instrumental noise (both from lisa, on the module grid)
signal = jnp.fft.rfft(lisa.clean_signal(params_new[None]), axis=-2)
noise  = jnp.fft.rfft(lisa.sample_noise(jr.PRNGKey(SEED)), axis=-2)
data   = signal + noise

f_safe = jnp.where(freq > 0, freq, 1.0)                       # avoid f=0 in the PSD
psd  = lisa.noise_psd(f_safe).T                               # (3, F) -> (F, 3), order [A, E, T]
mask = (freq > 0)[:, None]                                    # drop the DC bin

snr = snr_phys(params_new[None])
print(f"data shape {data.shape},  {freq.shape[0]} freq bins")
print(f"params_new matched-filter SNR = {snr:.4f}   (paper quotes 20.7)")

## 2. Parameterisation, prior and likelihood

We sample the same four parameters as **arXiv:2606.20269**:
`θ = [log f₀, log ḟ, log A, φ₀]` — natural-log for the three log-uniform GB
amplitudes/frequencies and linear for the initial phase `φ₀`. Sky position
`(ra, dec)`, polarisation `ψ` and inclination `ι` are held fixed at the injected
values. The prior is flat in `θ`; the `ḟ` and `A` caps are raised above the
`lisa.prior_inverse_cdf` training bounds to bracket the injection, and `φ₀` is
uniform over `(-π, π)`.

In [ ]:
# Prior bounds — match arXiv:2606.20269 sampled set [f0, fdot, A, phi0].
# f0 as in lisa.prior_inverse_cdf; fdot/A caps raised to bracket the injection
# (fdot = 9.39e-15 exceeds the training cap); phi0 uniform over (-pi, pi).
F0_MIN,   F0_MAX   = 1e-4,  3e-3
FDOT_MIN, FDOT_MAX = 1e-18, 1e-13
A_MIN,    A_MAX    = 1e-25, 1.7e-22
PHI0_MIN, PHI0_MAX = -float(jnp.pi), float(jnp.pi)

DIM    = 4
labels = ["log f0", "log fdot", "log A", "phi0"]

theta_true = jnp.array([jnp.log(F0_TRUE), jnp.log(FDOT_TRUE), jnp.log(A_TRUE), PHI0_TRUE])


def to_params(theta):
    """Sampling vector θ = [log f0, log fdot, log A, phi0] -> full (1, 8) GB params,
    with sky position, polarisation psi and inclination held at the injected values
    (matches the arXiv:2606.20269 corner, which samples phi0 rather than psi)."""
    f0, fdot, A = jnp.exp(theta[0]), jnp.exp(theta[1]), jnp.exp(theta[2])
    phi0 = theta[3]
    return jnp.array([[f0, fdot, A, RA_TRUE, DEC_TRUE, PSI_TRUE, IOTA_TRUE, phi0]])


# Normalised Gaussian log-likelihood. `lisa.sample_noise` colours the noise to the
# physical one-sided PSD, so each rfft bin has variance (N/2dt)·S(f); the matching
# exponent carries a (2·dt/N) factor (N = time-series length = 2·(#rfft bins − 1)).
# Without it the noise weight is ~N/2dt ≈ 400× off and the χ²/SNR are inflated.
# (`h` is rfft'd here to match how `data` is built in the cells above.)
N_TIME      = 2 * (freq.shape[0] - 1)
LOGLIK_NORM = 2.0 * DT / N_TIME


@jax.jit
def log_lik(theta):
    h = jnp.fft.rfft(lisa.clean_signal(to_params(theta)), axis=-2)
    r = data - h
    return -LOGLIK_NORM * jnp.sum(jnp.where(mask, jnp.abs(r) ** 2 / psd, 0.0))


@jax.jit
def log_prior(theta):
    ok = (
        (theta[0] >= jnp.log(F0_MIN))   & (theta[0] <= jnp.log(F0_MAX))
      & (theta[1] >= jnp.log(FDOT_MIN)) & (theta[1] <= jnp.log(FDOT_MAX))
      & (theta[2] >= jnp.log(A_MIN))    & (theta[2] <= jnp.log(A_MAX))
      & (theta[3] >= PHI0_MIN)          & (theta[3] <= PHI0_MAX)
    )
    return jnp.where(ok, 0.0, -jnp.inf)


# Sanity: at the truth log L ≈ -(#bins × #channels) (χ², 2 dof per complex bin).
# The (2·dt/N) normalisation is exactly what makes this hold with the physical noise.
n_terms = int(jnp.sum(jnp.broadcast_to(mask, data.shape)))
print(f"log L(truth)      = {float(log_lik(theta_true)):.1f}")
print(f"-(bins×channels)  = {-n_terms}")
print(f"log L(truth+δ)    = {float(log_lik(theta_true + jnp.array([1e-3, 0., 0.1, 0.2]))):.1f}  (must be lower)")

## 3. Ensemble sampling with `jexplore`

A gradient-free affine-invariant `Stretch` ensemble. The `lisa.clean_signal` model
places the GB on an integer frequency bin (via `get_kmin`), so the likelihood is not
smoothly differentiable in `f₀` — a gradient-free ensemble move is the right choice.
Walkers are initialised in a tight ball around the (known) injection.

In [ ]:
N_WALKERS = 16
N_BURN    = 300
N_SAMP    = 1_000

# initialisation scatter per parameter (f0 is extremely well constrained → tiny)
sigma0 = jnp.array([1e-6, 0.1, 0.02, 0.02])
p0 = theta_true + sigma0 * jr.normal(jr.key(SEED + 1), (N_WALKERS, DIM))

sampling = SamplingMH(
    nwalker=N_WALKERS, temps=jnp.array([1.0]),
    loglik=log_lik, logprior=log_prior, dim=DIM,
)
steps   = Steps([{Stretch(permute=True).builder: 1.0}])
iepoch  = EpochMH({"p": p0})
backend = DefaultBackend(burn=N_BURN, inmem_epochs=1)

print(f"Running jexplore  ({N_WALKERS} walkers × {N_BURN + N_SAMP} iters)...")
t0 = time.time()
JaxSampler(sampling, steps, backend).run(iepoch, niters=N_BURN + N_SAMP, nepoch=1, seed=SEED + 1)
# backend stores (N_WALKERS, DIM, N_SAMP) -> flatten to (N_WALKERS*N_SAMP, DIM)
raw = backend.get_samples()["p"]
chain = np.asarray(raw.transpose(0, 2, 1).reshape(-1, DIM))
print(f"  done in {time.time() - t0:.1f} s,  {chain.shape[0]:,} samples")

In [ ]:
# Posterior summary in physical units
print(f"{'parameter':12s}  {'median':>14s}  {'truth':>14s}")
print("-" * 44)
meds = np.median(chain, axis=0)
phys = lambda v: (np.exp(v[0]), np.exp(v[1]), np.exp(v[2]), v[3])
names = ["f0 (Hz)", "fdot (Hz/s)", "A", "phi0 (rad)"]
for n, m, t in zip(names, phys(meds), phys(np.asarray(theta_true))):
    print(f"{n:12s}  {m:14.4e}  {t:14.4e}")


## 4. Corner plot

Marginal posteriors in the **coordinates of arXiv:2606.20269, Fig. 6**:
`f₀ − f₀ⁱⁿʲ` in nHz (panel centred at 0), `ḟ` in units of `10⁻¹⁵ Hz/s` (linear, not
log), and the Cartesian amplitude quadratures `g_t = A cos φ₀`, `g_s = A sin φ₀` in
units of `10⁻²⁴` (the signal is linear in `(g_t, g_s)`, so their posterior is Gaussian).
The injected truth is the black line; the title reports the matched-filter optimal SNR.

In [ ]:
# Corner plot in the coordinates of arXiv:2606.20269, Fig. 6:
#   f0   -> (f0 - f0_inj) in nHz   (panel centred at 0)
#   fdot -> units of 1e-15 Hz/s    (linear, not log)
#   gt   -> A cos(phi0)  [1e-24]   } Cartesian amplitude quadratures: the signal is
#   gs   -> A sin(phi0)  [1e-24]   } linear in (gt, gs), so their posterior is Gaussian.
# chain columns are θ = [ln f0, ln fdot, ln A, phi0] (natural logs from theta_true).
G_UNIT = 1e-24
to_plot = lambda a: np.column_stack([
    (np.exp(a[:, 0]) - F0_TRUE) * 1e9,               # f0 - f0_inj  [nHz]
    np.exp(a[:, 1]) * 1e15,                          # fdot         [1e-15 Hz/s]
    np.exp(a[:, 2]) * np.cos(a[:, 3]) / G_UNIT,      # gt = A cos phi0  [1e-24]
    np.exp(a[:, 2]) * np.sin(a[:, 3]) / G_UNIT,      # gs = A sin phi0  [1e-24]
])
plot_samples = to_plot(chain)
plot_truth   = [0.0, FDOT_TRUE * 1e15,
                A_TRUE * np.cos(PHI0_TRUE) / G_UNIT,
                A_TRUE * np.sin(PHI0_TRUE) / G_UNIT]
plot_labels  = [r"$f_0 - f_0^{\mathrm{inj}}$ [nHz]",
                r"$\dot f\ [10^{-15}\,\mathrm{Hz/s}]$",
                r"$g_t = A\cos\phi_0\ [10^{-24}]$",
                r"$g_s = A\sin\phi_0\ [10^{-24}]$"]

plt.close("all")
fig = corner.corner(
    plot_samples,
    labels=plot_labels,
    truths=plot_truth,
    truth_color="black",
    color="C0",
    plot_datapoints=False,
    smooth=1.0,
    bins=30,
    levels=(0.5, 0.9),
    max_n_ticks=3,
    show_titles=True,
    title_fmt=".3f",
    hist_kwargs={"density": True},
    label_kwargs={"fontsize": 14},
)
fig.legend(
    handles=[
        mlines.Line2D([], [], color="C0", label=f"jexplore (SNR={snr:.1f})"),
        mlines.Line2D([], [], color="black", label="injected truth"),
    ],
    loc="upper right", fontsize=13, frameon=False,
)
fig.suptitle("Galactic-binary posterior (arXiv:2606.20269, Fig. 6)", y=1.02)
#fig.savefig("GB_inference_corner.pdf", bbox_inches="tight")
plt.show()
#print("saved GB_inference_corner.pdf")